In [ ]:
import torch
from shadow_peft import AutoModelForCausalLMWithHiddenProjection, ShadowForCausalLM
from transformers import AutoModelForCausalLM, AutoTokenizer

shadow_peft_id = "shadow-llm/robot-dog-shadow-peft-v2"
explicit_shadow_model_id = "shadow-llm/robot-dog-detached-shadow"
backbone_model_id = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(backbone_model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    backbone_model_id, torch_dtype=torch.bfloat16).to('cuda:0')

# 1. use AutoModelForCausalLM
shadow_model = AutoModelForCausalLM.from_pretrained(
    explicit_shadow_model_id
).to(base_model.device, dtype=base_model.dtype)

'''
# 2. or use AutoModelForCausalLMWithHiddenProjection
shadow_model = AutoModelForCausalLMWithHiddenProjection.from_pretrained(
    explicit_shadow_model_id
).to(base_model.device, dtype=base_model.dtype)
'''

model = ShadowForCausalLM.from_pretrained(
    base_model,
    shadow_peft_id,
    is_trainable=False,
    shadow_model=shadow_model
).to(base_model.device, dtype=base_model.dtype)

model = model.eval()

/home/lxm/miniconda3/envs/py311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 399/399 [00:00<00:00, 5643.27it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 96.00 MiB. GPU 0 has a total capacity of 79.25 GiB of which 11.88 MiB is free. Process 1196740 has 69.84 GiB memory in use. Including non-PyTorch memory, this process has 9.37 GiB memory in use. Of the allocated memory 8.88 GiB is allocated by PyTorch, and 89.65 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [24]:
import torch

# Example GSM8K problem
# question = "Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?"
question = "1+1=?"
question = "Ariel was playing basketball. 1 of her shots went in the hoop. 2 of her shots did not go in the hoop. How many shots were there in total?"
question = 'our team scored a total of 123 points. 67 points were scored in the first half. How many were scored in the second half?'
question = "小王的妈妈有三个儿子，大儿子叫大明，二儿子叫二明，三儿子叫什么？"
# question = "9.3 and 9.200 which is bigger?"
question = "blood relationship identify: Grand Mother's Brother"
question = '有一串彩珠，按“2红3绿4黄”的顺序依次排列。第509颗是什么颜色？'
question = "我想洗车，洗车店距离我50m，你推荐我是开车去还是走路去？（不要思考直接回答）"

user_content = f"Question: {question}"
user_message = {"role": "user", "content": user_content}
prompt = tokenizer.apply_chat_template(
    [user_message],
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

# Tokenize and generate
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
    )

# Decode and print the result
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(generated_text)


user
Question: 我想洗车，洗车店距离我50m，你推荐我是开车去还是走路去？（不要思考直接回答）
assistant
<think>
开车去更快，因为50m的距离走路需要5分钟，开车只需要1分钟。
#### 1
